## Example 1: aws_xray_sdk functionality test 1

In [ ]:
from aws_xray_sdk.core import xray_recorder, AWSXRayRecorder

# Example 1: Basic AWS X-Ray SDK setup
print("Example 1: Testing AWS X-Ray SDK basic setup")

try:
    # AWS X-Ray SDK for distributed tracing
    print("AWS X-Ray SDK imported successfully")
    
    # Check if recorder is available
    print(f"X-Ray recorder available: {xray_recorder is not None}")
    print(f"Recorder type: {type(xray_recorder).__name__}")
    assert isinstance(xray_recorder, AWSXRayRecorder), type(xray_recorder)
    
    # The global recorder exposes configuration and the segment API
    for method in ('configure', 'begin_segment', 'end_segment', 'begin_subsegment', 'end_subsegment', 'capture'):
        assert callable(getattr(xray_recorder, method)), method
    print("X-Ray recorder configuration and segment methods available")
    
    print("aws_xray_sdk example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"aws_xray_sdk example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: aws_xray_sdk functionality test 2

In [ ]:
import json
import re
from aws_xray_sdk.core import xray_recorder

# Example 2: X-Ray recorder configuration
print("Example 2: Testing X-Ray recorder configuration")


class CapturingEmitter:
    """Keeps finished segments in memory instead of sending them to the X-Ray daemon over UDP."""

    def __init__(self):
        self.entities = []

    def send_entity(self, entity):
        self.entities.append(json.loads(entity.serialize()))

    def set_daemon_address(self, address):
        pass


try:
    # Configure the recorder: sample every request and send segments to the in-memory emitter
    emitter = CapturingEmitter()
    xray_recorder.configure(
        context_missing='LOG_ERROR',
        service='test-service',
        sampling=False,
        emitter=emitter
    )
    assert xray_recorder.service == 'test-service', xray_recorder.service
    print("X-Ray recorder configured successfully")
    
    # Trace one request: a segment with one annotated subsegment
    xray_recorder.begin_segment('notebook-request')
    subsegment = xray_recorder.begin_subsegment('work')
    subsegment.put_annotation('items', 3)
    xray_recorder.end_subsegment()
    xray_recorder.end_segment()
    
    # The finished segment was emitted once, with its subsegment nested inside
    assert len(emitter.entities) == 1, emitter.entities
    trace = emitter.entities[0]
    print(f"Emitted segment: {trace['name']} (trace id {trace['trace_id']})")
    assert trace['name'] == 'notebook-request' and trace['in_progress'] is False, trace
    assert re.fullmatch(r"1-[0-9a-f]{8}-[0-9a-f]{24}", trace['trace_id']), trace['trace_id']
    [child] = trace['subsegments']
    assert child['name'] == 'work' and child['annotations'] == {'items': 3}, child
    assert child['parent_id'] == trace['id'] and child['trace_id'] == trace['trace_id'], child
    assert trace['start_time'] <= child['start_time'] <= child['end_time'] <= trace['end_time']
    print(f"Subsegment: {child['name']} with annotations {child['annotations']}")
    
    print("aws_xray_sdk example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"aws_xray_sdk example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: aws_xray_sdk functionality test 3

In [ ]:
from aws_xray_sdk.core import patch, patch_all, xray_recorder
from aws_xray_sdk.core.patcher import SUPPORTED_MODULES

# Example 3: X-Ray patching functionality
print("Example 3: Testing X-Ray patching functionality")

try:
    # patch_all() instruments every supported library that is installed; patch() picks some
    print(f"patch_all function available: {callable(patch_all)}")
    print(f"Supported modules: {SUPPORTED_MODULES}")
    assert {'botocore', 'requests', 'sqlite3'} <= set(SUPPORTED_MODULES), SUPPORTED_MODULES
    
    # Unknown module names are refused
    try:
        patch(['not_a_real_module'])
    except Exception as e:
        assert 'not supported' in str(e), e
        print(f"Unknown module refused: {e}")
    else:
        raise AssertionError("patch() accepted an unknown module")
    
    # Patch only sqlite3 from the standard library. This changes sqlite3.connect for the rest of
    # this kernel session: new connections are traced. CapturingEmitter comes from Example 2.
    emitter = CapturingEmitter()
    xray_recorder.configure(emitter=emitter, sampling=False)
    patch(['sqlite3'])
    import sqlite3
    
    xray_recorder.begin_segment('db-request')
    connection = sqlite3.connect(':memory:')
    row = connection.execute('select 1 + 1').fetchone()
    connection.close()
    xray_recorder.end_segment()
    assert row == (2,), row
    
    # The query was recorded as a remote subsegment with SQL metadata
    [trace] = emitter.entities
    [query] = trace['subsegments']
    print(f"Recorded query subsegment: {query['name']} {query['sql']}")
    assert query['namespace'] == 'remote', query
    assert query['sql']['database_type'] == 'sqlite3', query
    assert query['sql']['database_version'] == sqlite3.sqlite_version, query
    
    print("aws_xray_sdk example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"aws_xray_sdk example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: aws_xray_sdk functionality test 4

In [ ]:
import json
import re
from aws_xray_sdk.core.models.segment import Segment
from aws_xray_sdk.core.models.subsegment import Subsegment

# Example 4: X-Ray Segment creation
print("Example 4: Testing X-Ray Segment creation")

try:
    # Test creating X-Ray segments for tracing
    print("Testing X-Ray Segment creation")
    
    # Create a basic segment
    segment = Segment('test-segment')
    print("X-Ray segment created successfully")
    print(f"Segment name: {segment.name}")
    print(f"Segment type: {type(segment).__name__}")
    assert segment.name == 'test-segment' and segment.in_progress and segment.sampled
    assert re.fullmatch(r"1-[0-9a-f]{8}-[0-9a-f]{24}", segment.trace_id), segment.trace_id
    assert re.fullmatch(r"[0-9a-f]{16}", segment.id), segment.id
    
    # Annotations are indexed for search, metadata is not; a subsegment records one unit of work
    segment.put_annotation('customer_tier', 'gold')
    segment.put_metadata('request', {'items': 2}, 'orders')
    child = Subsegment('lookup', 'local', segment)
    segment.add_subsegment(child)
    child.close()
    segment.close()
    assert not segment.in_progress and segment.end_time >= segment.start_time
    
    # The JSON document that would be sent to X-Ray
    document = json.loads(segment.serialize())
    print(f"Serialized segment keys: {sorted(document)}")
    assert document['annotations'] == {'customer_tier': 'gold'}, document
    assert document['metadata'] == {'orders': {'request': {'items': 2}}}, document
    [sub_document] = document['subsegments']
    assert sub_document['name'] == 'lookup' and sub_document['parent_id'] == segment.id, sub_document
    print("Segment with annotation, metadata and subsegment serialized")
    
    print("aws_xray_sdk example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"aws_xray_sdk example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: aws_xray_sdk functionality test 5

In [ ]:
import importlib
from importlib.metadata import version as distribution_version
import aws_xray_sdk
from aws_xray_sdk.version import VERSION

# Example 5: AWS X-Ray SDK package validation
print("Example 5: AWS X-Ray SDK package validation")

try:
    # Comprehensive validation of AWS X-Ray SDK package
    print("AWS X-Ray SDK package validation")
    
    # The package's version module agrees with the installed distribution
    print(f"AWS X-Ray SDK version: {VERSION}")
    assert VERSION == distribution_version("aws-xray-sdk"), VERSION
    
    # Main modules: a missing one fails the cell
    main_modules = [
        'aws_xray_sdk.core',
        'aws_xray_sdk.core.models.segment',
        'aws_xray_sdk.core.models.subsegment',
        'aws_xray_sdk.core.context',
        'aws_xray_sdk.core.sampling.local.sampler',
        'aws_xray_sdk.core.emitters.udp_emitter',
    ]
    available_modules = [importlib.import_module(name).__name__ for name in main_modules]
    assert available_modules == main_modules, available_modules
    
    print(f"Available modules: {available_modules}")
    print("AWS X-Ray SDK provides distributed tracing for AWS applications")
    
    print("aws_xray_sdk example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"aws_xray_sdk example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")